# Official Bolbosh Kashmiri TTS evaluation
Evaluates the official GAASH-Lab native-script checkpoint with its male and female RASA speakers. No training or deployment.


In [ ]:
!nvidia-smi
!git clone -q --depth 1 --filter=blob:none --sparse https://github.com/gaash-lab/Bolbosh.git /kaggle/working/Bolbosh
%cd /kaggle/working/Bolbosh
!git sparse-checkout set matcha configs setup.py pyproject.toml requirements.txt README.md
!pip install -q --no-cache-dir 'numpy<2' 'cython>=3'
!pip install -q --no-cache-dir --no-build-isolation -e .
!python setup.py build_ext --inplace -q
!python -c "from matcha.models.matcha_tts import MatchaTTS; print('Official Bolbosh code ready')"


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
checkpoint=hf_hub_download('GAASH-Lab/Bolbosh-checkpoint','model.ckpt',repo_type='model',local_dir='/kaggle/working/bolbosh-model')
assert Path(checkpoint).stat().st_size > 200_000_000
sentences=[
 'اَلسَلامُ عَلَیکُم۔ تُہۍ کِتھ پٲٹھۍ چھِو؟',
 'مےٚ ناو چھُ سمیر۔ بہٕ چھُس کویل پُلوامہٕ پؠٹھ۔',
 'کٲشُر زَبان چھِ واریاہ خوٗبصوٗرَت۔',
 'اَسہِ گَژھِ صاف تہٕ صحیح کٲشُر بولُن۔',
]
text_file=Path('/kaggle/working/bolbosh-review-sentences.txt')
text_file.write_text('\n'.join(sentences)+'\n',encoding='utf-8')
print('Checkpoint:',checkpoint,'bytes:',Path(checkpoint).stat().st_size)


In [ ]:
import subprocess, sys
for speaker,label in ((422,'male'),(423,'female')):
    output=f'/kaggle/working/bolbosh-{label}'
    cmd=[sys.executable,'-m','matcha.cli','--checkpoint_path',checkpoint,'--vocoder','hifigan_univ_v1','--file',str(text_file),'--spk',str(speaker),'--steps','20','--speaking_rate','1.0','--output_folder',output]
    result=subprocess.run(cmd,text=True,capture_output=True,timeout=3600)
    print(result.stdout[-12000:]); print(result.stderr[-12000:])
    if result.returncode: raise RuntimeError(f'Bolbosh {label} inference failed: {result.returncode}')


In [ ]:
import json, zipfile
review=Path('/kaggle/working/bolbosh-official-review'); review.mkdir(exist_ok=True)
manifest=[]
for label,speaker in (('male',422),('female',423)):
    generated=sorted(Path(f'/kaggle/working/bolbosh-{label}').glob('*.wav'))
    assert len(generated)==len(sentences), generated
    for index,(audio,text) in enumerate(zip(generated,sentences),1):
        target=review/f'{label}-{index:02d}.wav'; target.write_bytes(audio.read_bytes())
        manifest.append({'file':target.name,'speaker':speaker,'voice':label,'kashmiri':text})
(review/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
(review/'ATTRIBUTION.txt').write_text('Bolbosh by GAASH-Lab. Model/code: MIT. Dataset: CC BY 4.0 with upstream attribution. Paper: arXiv:2603.07513.\n',encoding='utf-8')
archive=Path('/kaggle/working/bolbosh-official-listening-samples.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in review.iterdir(): z.write(path,path.name)
print('Review artifact:',archive,archive.stat().st_size,'bytes')
